# Data Decomposition
This exercise will walk you through some basics of data decompositions. We will be thinking of our data from the perspective of frequency transforms and matrix decompositions.

We are providing this notebook as 

Please try to go through CSHL2025_DataProcessingExercises.ipynb on your own before referring to this notebook. 

In [ ]:
#########################################################################
## This exercise will walk you though some basics of data decompositions

## I like to keep all the imports at the top of the file
from skimage import io  # skimage is a package for dealing with images. io = in/out
import numpy as np      # numpy is the core package for numerical processing
import scipy.io
import h5py             # This package allows you to access newer MATLAB saves that are in HDF5 formats
from matplotlib import pyplot   
import plotly.express as px


In [ ]:
#########################################################################
## Let's first load the data

matFileData = h5py.File('/home/aaa_shared/cshl2025/Data/Somatic/SEUDO_J122_2015-11-16_L01_c01_b27.mat', mode='r')

matFileData['dFF'].shape
dataMovie = np.array(matFileData['dFF'])
dMshape   = dataMovie.shape # Keep the orignial size!!
dMshape

In [ ]:
#########################################################################
## Let's look at the frequency transform of a single pixel. 

# a) Pick a pixel that you feel has a lot of signal (look at the movie!)
# b) Pick a pixel that you feel has minimal signal

# To identify these points, let's start by looking at a summary image
#summaryImage = np.median(dataMovie,axis=0)
summaryImage = np.mean(dataMovie,axis=0)
fig = px.imshow(summaryImage,zmin=-0.1,zmax=0)  
fig.show()

In [ ]:
# for a) it looks like location (12,25) is pretty much mid-neuron
# for b) it looks like location (23,60) is pretty dark

# Lets plot the two to make sure
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(range(dataMovie.shape[0])), y=dataMovie[1000:3000,60,23],
                    mode='lines',name='noise'))
fig.add_trace(go.Scatter(x=np.array(range(dataMovie.shape[0])), y=dataMovie[1000:3000,25,12],
                    mode='lines',name='signal'))
fig.show()

In [ ]:
# c) Compute the Fourier transoforms of both time traces
#         - Hint: look up numpy.fft and see how to use it

# Now let's calculate the fourier coefficients
traceSignal = dataMovie[:,25,12] # Extract the signal trace
traceNoise  = dataMovie[:,60,23] # Extract the noise trace

signalFFT = np.fft.fftshift(np.fft.fft(traceSignal))  # Take the Fourier transform of the signal trace
noiseFFT  = np.fft.fftshift(np.fft.fft(traceNoise))   # Take the Fourier transform of the noise trace

# Often we plot the "power" of the data which corresponds to 
# the magnitude of the different frequency components. This is 
# done with the np.abs() function which takes the absolute value
# of an imaginary number (sqrt(x^2 + y^2) for x + j*y)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(range(len(noiseFFT))), y=np.log(np.abs(noiseFFT)),
                    mode='lines',name='noise'))
fig.add_trace(go.Scatter(x=np.array(range(len(signalFFT))), y=np.log(np.abs(signalFFT)),
                    mode='lines',name='signal'))
fig.show()

In [1]:
#########################################################################
# d) Repeat the above but this time use your code from yesterday to average
#    over a cell that does have signal, and over a "blank" area

# First get some traces that are local averages of the above traces:
traceSignalAvg = np.mean(dataMovie[:,2:30,7:17],axis=(1,2))   # Extract the average signal trace
traceNoiseAvg  = np.mean(dataMovie[:,55:65,18:28],axis=(1,2)) # Extract the average noise trace

signalFFTAvg = np.fft.fftshift(np.fft.fft(traceSignalAvg))  # Take the Fourier transform of the signal trace
noiseFFTAvg  = np.fft.fftshift(np.fft.fft(traceNoiseAvg))   # Take the Fourier transform of the noise trace

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(range(len(noiseFFTAvg))), y=np.log(np.abs(noiseFFT)),
                    mode='lines',name='noise'))
fig.add_trace(go.Scatter(x=np.array(range(len(signalFFTAvg))), y=np.log(np.abs(signalFFT)),
                    mode='lines',name='signal'))
fig.show()


NameError: name 'np' is not defined

In [ ]:
# Questions:
# 1) What do you notice about the Fourier transform values?
# 2) How do the Fourier represenrtations change between both time-traces? 


In [ ]:
#########################################################################
# Now let's try out some data-driven decompositions

# a) Start with PCA: Take the first 200 frames of the SEUDO dataset from
#    yesterday and perform PCA on it. Hint: This data is 3D!? Try running
#    PCA using different numbers of principal components.  
#      - Hint: look up sklearn.decomposition.PCA

import sklearn as skl
import sklearn.decomposition as dcmp

# First we need to take the 3D data and make it a 2D matrix that is time-by-pixels 
# OR pixels-by-time. Since time is already the 0 axis, let's instead reorder the data
# so that the d
dataMovie = np.reshape(dataMovie, (dMshape[0],dMshape[1]*dMshape[2]))

# Now create a PCA object that takes the first 50 components of the data
dataPCA   = dcmp.PCA(n_components=50);   # This line makes the PCA object
dataPCA.fit(dataMovie)                   # This line computes the PCA for the dataMovie array
dataMovie = np.reshape(dataMovie,dMshape)# Reshape the movie back in case we want to use it later

# Now let's look at the PCs. First of all, note that the PCA components are in the reshaped
# space so we need to reshape them back!
PCA_compImgs = np.reshape(dataPCA.components_,(50,dMshape[1],dMshape[2]))

# Now let's take a look at the spatial PC components:
fig = px.imshow(PCA_compImgs, animation_frame=0, binary_string=True, labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
cnts = pyplot.hist(np.ravel(dataMovie),bins=2000)

In [ ]:
# b) Now use a version of matrix decompositions we didn't cover: NMF. 
#    NMF stands for non-negative matrix decompositions. It's the basis for
#    some of the more well known decomposition algorithms. Use this on the
#    same data as above. 
#      - Hint: look up sklearn.decomposition.NMF

# so that the d
dataMovie = np.reshape(dataMovie, (dMshape[0],dMshape[1]*dMshape[2]))
dataMovie2 = dataMovie
dataMovie2[dataMovie<0] = 0
# Now create a NMF object that takes the first 50 components of the data
dataNMF   = dcmp.NMF(n_components=10,max_iter=100);   # This line makes the PCA object
dataNMF.fit_transform(dataMovie2)        # This line computes the PCA for the dataMovie array
dataMovie = np.reshape(dataMovie,dMshape)# Reshape the movie back in case we want to use it later

del dataMovie2

NMF_compImgs = np.reshape(dataNMF.components_,(10,dMshape[1],dMshape[2]))

# Now let's take a look at the spatial NMF components:
fig = px.imshow(NMF_compImgs, animation_frame=0, binary_string=True, labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
# c) Finally, try out ICA! Look through the sklearn.decomposition 
#    documentation. Is there an implementation that you can use? 


dataICA = dcmp.FastICA(n_components=7, random_state=0, whiten='unit-variance')

dataMovie = np.reshape(dataMovie, (dMshape[0],dMshape[1]*dMshape[2]))
X_transformed = dataICA.fit_transform(dataMovie)
X_transformed.shape
dataMovie = np.reshape(dataMovie,dMshape)# Reshape the movie back in case we want to use it later


In [ ]:
## Now let's plot the ICA components!!
ICA_compImgs = np.reshape(dataICA.components_,(7,dMshape[1],dMshape[2])) # First reshape the images into actual images from vectors

# Now let's take a look at the spatial ICA components:
fig = px.imshow(ICA_compImgs, animation_frame=0, binary_string=True, labels=dict(animation_frame="slice"))
fig.show()


In [ ]:
# Questions
# 1) What does the first element of the PCA output look like? The second? The last?
# 2) How do the outputs of the three algorithms compare? 
# 3) Try to "align" the outputs by matching up ouputs (decomposition vectors)
#    between the threen algorithms.